This is the Python script that leverages the sagemaker.processing.ScriptProcessor SDK to spin up an isolated compute node and mounts your raw S3 bucket prefixes in FastFile mode.

In [ ]:
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

def launch_isolated_fastfile_job():
    print("=== Initializing SageMaker Staging Session ===")
    
    # 1. Establish secure IAM credentials from your active SageMaker Domain environment
    sagemaker_session = sagemaker.Session()
    role = sagemaker.get_execution_role()
    
    # Define your challenge storage infrastructure buckets
    # (Ensure these match your actual bucket naming configurations)
    bucket_name = "your-challenge-s3-bucket-name"
    s3_input_uri = f"s3://{bucket_name}/raw-data/"
    s3_output_uri = f"s3://{bucket_name}/candidates/"
    
    # 2. Provision the isolated, cost-efficient compute engine node
    # It includes up to 50 GB of volatile ephemeral scratch storage that wipes post-run
    processor = ScriptProcessor(
        role=role,
        image_uri=sagemaker.image_uris.retrieve(framework="scikit-learn", region=sagemaker_session.boto_region_name, version="1.2-1"),
        command=["python3"],
        instance_count=1,
        instance_type="ml.m5.xlarge",  # Free-tier eligible compute hours matrix
        volume_size_in_gb=50           # Ephemeral workspace partition bounds
    )
    
    print(f"-> Launching Isolated Compute Node with FastFile Mount targeting: {s3_input_uri}")
    
    # 3. Trigger execution sequence and link the data communication streams
    processor.run(
        # The processing pipeline logic script file extracted from your notebook
        code="../../Package/code/business_entity_resolution/src/blocking_pipeline.py",
        
        inputs=[
            ProcessingInput(
                source=s3_input_uri,
                destination="/opt/ml/processing/input",
                input_mode="FastFile"  # Stream rows dynamically from S3; zero disk footprint!
            )
        ],
        outputs=[
            ProcessingOutput(
                source="/opt/ml/processing/output",
                destination=s3_output_uri,
                output_name="candidate_pairs"
            )
        ]
    )
    print("=== Isolated Staging Compute Run Concluded Cleanly ===")

if __name__ == "__main__":
    launch_isolated_fastfile_job()
